# 🧪 Лабораторна робота №1
## Тема: Основи MicroPython на ESP32 — цифрові входи/виходи (GPIO)

## 🎯 Мета роботи:
- Ознайомитися із середовищем MicroPython та способами запуску коду на ESP32
- Навчитися керувати цифровими виходами (`Pin.OUT`) — вмикати/вимикати світлодіоди
- Навчитися зчитувати цифрові входи (`Pin.IN`) — кнопки, підтягувальні резистори
- Реалізувати усунення брязкоту (debounce) та прості патерни блимання

---

## 📖 Розширені теоретичні відомості

### 1️⃣ Що таке ESP32 і MicroPython
- **ESP32** — мікроконтролер з Wi-Fi та Bluetooth, двома ядрами, десятками пінів GPIO, вбудованими АЦП, ЦАП, PWM, I2C, SPI, UART.
- **MicroPython** — компактна реалізація Python 3, що виконується безпосередньо на мікроконтролері (без ОС).
- На відміну від звичайного Python, код виконується **на платі**, а не на комп'ютері — комп'ютер лише надсилає команди через USB (Serial) або Wi-Fi (WebREPL).
- Після ввімкнення плата запускає `boot.py`, а потім `main.py` — файли, що зберігаються у власній файловій системі плати.

### 2️⃣ Підключення та прошивка
- **Прошивка**: одноразово записується файл MicroPython `.bin` на плату утилітою `esptool.py` (`esptool.py --port COM5 write_flash -z 0x1000 firmware.bin`).
- **Робота з платою** можлива кількома способами (див. таблицю нижче).

| Спосіб | Опис |
|---|---|
| Thonny IDE | Найпростіший спосіб для початківців: вибір плати, редагування й запуск `main.py` |
| `mpremote` / `ampy` | Консольні утиліти для копіювання файлів і запуску REPL |
| Jupyter + `jupyter_micropython_kernel` | Виконання коду по комірках прямо в ноутбуці (див. розділ 8️⃣) |
| Wokwi (wokwi.com) | Онлайн-симулятор ESP32 у браузері, без фізичної плати |

- REPL (Read-Eval-Print Loop) — інтерактивна консоль плати, доступна через послідовний порт (Serial), аналог консолі Python.

### 3️⃣ Клас `machine.Pin`
- Керування ніжкою (пином) плати відбувається через об'єкт `Pin`:
  ```python
  from machine import Pin

  led = Pin(2, Pin.OUT)          # пін 2 як вихід (на більшості плат — вбудований LED)
  button = Pin(4, Pin.IN, Pin.PULL_UP)  # пін 4 як вхід з підтяжкою до +3.3V
  ```

- Режими: `Pin.OUT` (вихід), `Pin.IN` (вхід), `Pin.PULL_UP` / `Pin.PULL_DOWN` (внутрішня підтяжка).
- Методи виходу: `led.value(1)` / `led.value(0)`, а також зручні `led.on()`, `led.off()`, `led.toggle()`.
- Читання входу: `button.value()` — поверне `0` або `1`.
- **Важливо:** кнопка з `PULL_UP` у неактивному стані дає `1`, а при натисканні (замикання на GND) — `0`.

### 4️⃣ Затримки часу
| Функція | Опис |
|---|---|
| `time.sleep(1)` | пауза 1 секунда |
| `time.sleep_ms(500)` | пауза 500 мілісекунд |
| `time.sleep_us(200)` | пауза 200 мікросекунд |
| `time.ticks_ms()` | "мітка часу" в мс — використовується для вимірювання інтервалів без блокування |
| `time.ticks_diff(t2, t1)` | коректна різниця між двома мітками (враховує переповнення лічильника) |

`time.sleep()` **блокує** виконання — поки він триває, плата нічого іншого не робить. Це нормально для простих завдань, але в Лабораторній №3 ми навчимося уникати блокування за допомогою таймерів і переривань.

### 5️⃣ Цифровий вихід — керування світлодіодом
Найпростіша програма — блимання (Blink):
```python
from machine import Pin
import time

led = Pin(2, Pin.OUT)
while True:
    led.value(1)
    time.sleep(0.5)
    led.value(0)
    time.sleep(0.5)
```
На платі DevKit більшості виробників вбудований світлодіод підключений до `GPIO2`. У Wokwi зазвичай додають зовнішній LED з резистором ~220 Ом до довільного піна (наприклад `GPIO2`).

### 6️⃣ Цифровий вхід — кнопки та усунення брязкоту
- Механічна кнопка при натисканні/відпусканні кілька мілісекунд "дрижить" (bounce) — контакт кілька разів замикається й розмикається, що комп'ютер бачить як кілька натискань замість одного.
- Найпростіший спосіб усунути брязкіт — коротка пауза після зміни стану:
  ```python
  from machine import Pin
  import time

  button = Pin(4, Pin.IN, Pin.PULL_UP)
  prev = 1
  while True:
      cur = button.value()
      if cur == 0 and prev == 1:      # спрацювало натискання (перепад 1 -> 0)
          print("Натиснуто!")
          time.sleep_ms(50)           # пауза для усунення брязкоту
      prev = cur
  ```

### 7️⃣ Типові помилки початківців
- Забули імпортувати `Pin` з `machine` → `NameError`.
- Використали пін, якого немає на платі, або зарезервований (наприклад `GPIO6-11` зайняті flash-пам'яттю) → плата не запускається.
- Підключили кнопку без резистора й без `PULL_UP/PULL_DOWN` → вхід "висить у повітрі" (плаваюче значення).
- Забули усунення брязкоту → одне натискання зараховується як кілька.
- Плутають логіку `PULL_UP` (натиснуто = `0`) з `PULL_DOWN` (натиснуто = `1`).
- Нескінченний `while True` без жодної затримки — плата "зависає" в дуже швидкому циклі й REPL стає непридатним для переривання (`Ctrl+C` у Serial-консолі, щоб зупинити).

### 8️⃣ Як перевіряти код цієї лабораторної роботи

У MicroPython немає локального "пісочного" середовища, як для звичайного Python — код виконується **на самому мікроконтролері** ESP32. Є два способи перевірити свої рішення.

**Варіант A. Jupyter Notebook + реальна плата ESP32 (рекомендовано, якщо є плата)**

1. Прошийте плату прошивкою MicroPython ([https://micropython.org/download/ESP32_GENERIC/](https://micropython.org/download/ESP32_GENERIC/)), один раз через `esptool.py`.
2. Встановіть ядро для Jupyter, яке вміє працювати з платою по USB/Serial:
   ```bash
   pip install jupyter_micropython_kernel
   python -m jupyter_micropython_kernel.install
   ```

3. Відкрийте цей ноутбук і оберіть ядро **MicroPython** (або **MicroPython - USB**) замість Python 3.
4. У першій комірці виконайте підключення до плати (замініть порт на свій — `COMx` у Windows, `/dev/ttyUSB0` або `/dev/ttyACM0` у Linux/macOS):
   ```
   %serialconnect --port=COM5 --baud=115200
   ```

5. Далі виконуйте комірки з кодом як завжди — код одразу запускається на платі, а `print()` повертає результат прямо в ноутбук. Світлодіоди й кнопки реагують фізично.
6. Якщо потрібно скинути з'єднання: `%disconnect`; список файлів на платі показує `%ls`, а перелік усіх доступних магічних команд ядра — `%lsmagic`.

**Варіант B. Немає плати під рукою — симулятор Wokwi**

1. Відкрийте [https://wokwi.com](https://wokwi.com) → **New Project → ESP32 → MicroPython**.
2. У файлі `main.py` вставте код із завдання.
3. На вкладці **diagram.json** опишіть схему (перелік деталей і з'єднань наведено в кожному завданні нижче) — або зберіть її мишкою в графічному редакторі Wokwi, перетягуючи компоненти з панелі **Parts**.
4. Натисніть зелену кнопку **▶ Play** — почнеться симуляція. Вивід `print()` з'являється у вкладці **Serial Monitor**.
5. Перевірте результат за критеріями, вказаними у кожному завданні (що саме має блимати/показуватись/виводитись).
6. Готовий проєкт можна зберегти (**Save**) і надіслати викладачу посилання на нього — це і буде звіт з лабораторної.

**Схема для цієї лабораторної (Wokwi):** ESP32 DevKit-C, світлодіод (LED) з резистором 220 Ом на `GPIO2`, друга пара LED+резистор на `GPIO4`/`GPIO5` для завдань із кількома LED, кнопка (Pushbutton) на `GPIO15` (інший вивід кнопки — на GND, підтяжку `PULL_UP` вмикаємо програмно, зовнішній резистор не обов'язковий).

> 💡 Порада: пишіть і перевіряйте код невеликими фрагментами — спочатку переконайтесь, що плата взагалі відповідає (наприклад, `print("hello")`), і лише потім додавайте логіку.

---

## 📝 Завдання 1. Блимання вбудованим світлодіодом
Підключіть світлодіод до `GPIO2` (або використайте вбудований) і напишіть програму, яка блимає ним з періодом 1 секунда (0.5 с увімкнено, 0.5 с вимкнено).

In [ ]:
from machine import Pin
import time

led = Pin(2, Pin.OUT)

while True:
    led.value(1)
    time.sleep(0.5)
    led.value(0)
    time.sleep(0.5)


### 🔢 Завдання:
1. Зменшіть період блимання до 0.2 с.
2. Перепишіть код, використовуючи `led.on()` / `led.off()` замість `value()`.
3. Перепишіть код, використовуючи `led.toggle()` в одному місці циклу.
4. Зробіть блимання рівно 10 разів, після чого зупиніться (цикл `for`).
5. Реалізуйте блимання за принципом «швидко-швидко-повільно» (3 короткі, 1 довгий спалах).
6. Додайте другий світлодіод на `GPIO4`, який блимає у протифазі до першого.
7. Зробіть так, щоб два світлодіоди блимали по черзі з різною швидкістю (1-й — 0.3 с, 2-й — 0.7 с).
8. Напишіть функцію `blink(pin, times, delay)`, яка блимає задану кількість разів.
9. Виведіть у Serial (`print`) номер поточного спалаху.
10. Закодуйте азбукою Морзе слово `SOS` (короткий-короткий-короткий, довгий-довгий-довгий, короткий-короткий-короткий).

## 📝 Завдання 2. Кнопка керує світлодіодом
Підключіть кнопку до `GPIO15` (з `PULL_UP`) і світлодіод до `GPIO2`. Світлодіод повинен світитися, доки кнопка утримується натиснутою.

In [ ]:
from machine import Pin
import time

led = Pin(2, Pin.OUT)
button = Pin(15, Pin.IN, Pin.PULL_UP)

while True:
    if button.value() == 0:     # кнопка натиснута (замкнута на GND)
        led.on()
    else:
        led.off()
    time.sleep_ms(20)


### 🔢 Завдання:
1. Інвертуйте логіку: LED світиться, доки кнопка НЕ натиснута.
2. Зробіть так, щоб одне коротке натискання перемикало стан LED (toggle), а не утримувало його.
3. Додайте другу кнопку, яка вмикає другий світлодіод незалежно від першої.
4. Реалізуйте логіку «І»: LED вмикається, лише якщо натиснуті ОБИДВІ кнопки одночасно.
5. Реалізуйте логіку «АБО»: LED вмикається, якщо натиснута хоча б одна з двох кнопок.
6. Додайте виведення в Serial повідомлення «Кнопка натиснута» / «Кнопка відпущена» лише при зміні стану (без повторів).
7. Зробіть так, щоб довге утримання (> 1 с) вмикало інший LED, ніж коротке натискання.
8. Використайте `Pin.PULL_DOWN` та зовнішній резистор — перепишіть логіку під нову схему.
9. Додайте світлодіод, що блимає, поки кнопка НЕ натиснута, і горить постійно, коли натиснута.
10. Виведіть у Serial поточний стан кнопки кожну секунду незалежно від натискань.

## 📝 Завдання 3. Усунення брязкоту і лічильник натискань
Реалізуйте коректний підрахунок натискань кнопки (без «фантомних» подвійних спрацювань) і виводьте лічильник у Serial після кожного натискання.

In [ ]:
from machine import Pin
import time

button = Pin(15, Pin.IN, Pin.PULL_UP)
count = 0
prev = 1

while True:
    cur = button.value()
    if cur == 0 and prev == 1:      # перепад 1 -> 0 = натискання
        time.sleep_ms(30)           # усунення брязкоту
        if button.value() == 0:     # підтверджуємо, що це не «дрижання»
            count += 1
            print("Натискань:", count)
    prev = cur
    time.sleep_ms(5)


### 🔢 Завдання:
1. Змініть логіку так, щоб рахувались відпускання кнопки, а не натискання.
2. Додайте світлодіод, що вмикається після кожного 5-го натискання й одразу вимикається.
3. Реалізуйте скидання лічильника на нуль після утримання кнопки довше 2 секунд.
4. Порівняйте результат з усуненням брязкоту і без нього: тимчасово приберіть `sleep_ms(30)` і поясніть у коментарі, що змінилось.
5. Виміряйте час між двома послідовними натисканнями за допомогою `time.ticks_ms()` і виведіть його.
6. Зробіть так, щоб швидкість блимання LED залежала від лічильника натискань (чим більше — тим швидше).
7. Додайте другу кнопку-лічильник і виводьте різницю між двома лічильниками.
8. Обмежте лічильник максимумом 10 — після 10-го натискання подальші ігноруються.
9. Реалізуйте «подвійне клацання» (double-click): окрема дія, якщо два натискання відбулись менш ніж за 400 мс.
10. Збережіть лічильник у список останніх 5 часових міток натискань і виведіть його.

## 📝 Завдання 4. «Біжучий вогник» на кількох світлодіодах
Підключіть 4 світлодіоди (наприклад до `GPIO2, 4, 5, 18`) і реалізуйте ефект «біжучого вогника»: по черзі запалюється один світлодіод, інші згасають.

In [ ]:
from machine import Pin
import time

pins = [Pin(p, Pin.OUT) for p in (2, 4, 5, 18)]

def all_off():
    for p in pins:
        p.off()

while True:
    for p in pins:
        all_off()
        p.on()
        time.sleep(0.2)


### 🔢 Завдання:
1. Зробіть рух у зворотному напрямку (справа наліво).
2. Реалізуйте рух «туди-сюди» (як біжучий вогник Larson Scanner / KITT).
3. Додайте кнопку, що змінює напрямок руху на протилежний.
4. Додайте другу кнопку, що змінює швидкість (затримку) руху.
5. Замість одного увімкненого світлодіода зробіть «хвіст» із двох сусідніх, що згасають поступово (простим `on/off`, без ШІМ).
6. Реалізуйте бінарний лічильник від 0 до 15 на 4 світлодіодах (кожен LED — один біт).
7. Зробіть так, щоб при досягненні останнього світлодіода все почалось спочатку з коротким миготінням усіх LED.
8. Винесіть логіку в функцію `running_light(pins, delay)`.
9. Додайте п'ятий світлодіод і адаптуйте код без дублювання.
10. Виведіть у Serial номер поточного активного світлодіода.

## 📝 Завдання 5. Міні-проєкт: світлофор
Реалізуйте модель світлофора на трьох світлодіодах (червоний — `GPIO2`, жовтий — `GPIO4`, зелений — `GPIO5`) з коректними інтервалами: червоний 3 с → червоний+жовтий 1 с → зелений 3 с → жовтий 1 с → знову червоний.

In [ ]:
from machine import Pin
import time

red = Pin(2, Pin.OUT)
yellow = Pin(4, Pin.OUT)
green = Pin(5, Pin.OUT)

def all_off():
    red.off(); yellow.off(); green.off()

while True:
    all_off(); red.on()
    time.sleep(3)

    yellow.on()             # червоний + жовтий залишаються
    time.sleep(1)

    all_off(); green.on()
    time.sleep(3)

    all_off(); yellow.on()
    time.sleep(1)


### 🔢 Завдання:
1. Додайте кнопку «пішохідного переходу», яка позачергово перемикає світлофор на червоний.
2. Додайте другий комплект світлодіодів для пішохідного світлофора (червоний/зелений чоловічок), синхронізований з основним.
3. Зробіть так, щоб зелений перед вимкненням «мигав» 3 рази (попередження про закінчення часу).
4. Виведіть у Serial назву поточної фази («Червоний», «Зелений» тощо) і час, що залишився.
5. Винесіть тривалості фаз у змінні на початку файлу, щоб їх було легко змінювати.
6. Реалізуйте «нічний режим» (кнопкою) — жовтий блимає раз на секунду, інші вимкнені.
7. Додайте лічильник повних циклів світлофора від старту програми.
8. Реалізуйте два незалежні світлофори на різних пінах, що працюють у протифазі (як на перехресті).
9. Перепишіть логіку фаз через список кортежів `[(колір, тривалість), ...]` і цикл `for`.
10. Додайте аварійний режим (кнопка) — усі три світлодіоди блимають одночасно раз на 0.5 с.

---

## Висновок

> В цій ...